# Automatic Number Plate Recognition (ANPR) using CNN

This notebook demonstrates Automatic Number Plate Recognition using Computer Vision and CNN techniques.

In [ ]:
# Install required libraries (if needed)
# !pip install opencv-python tensorflow matplotlib numpy easyocr


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator


## Load and Display Image

In [ ]:
image_path = 'car.jpg'   # Replace with your image path

image = cv2.imread(image_path)

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10,6))
plt.imshow(image_rgb)
plt.title("Input Image")
plt.axis('off')
plt.show()


## Convert to Grayscale and Detect Plate

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

plate_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_russian_plate_number.xml')

plates = plate_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4)

for (x, y, w, h) in plates:
    cv2.rectangle(image_rgb, (x, y), (x+w, y+h), (0,255,0), 2)

plt.figure(figsize=(10,6))
plt.imshow(image_rgb)
plt.title("Detected Number Plate")
plt.axis('off')
plt.show()


## Crop Plate Region

In [ ]:
for (x, y, w, h) in plates:
    plate = gray[y:y+h, x:x+w]

plt.figure(figsize=(8,4))
plt.imshow(plate, cmap='gray')
plt.title("Cropped Plate")
plt.axis('off')
plt.show()


## CNN Model for Character Recognition

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,1)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(36, activation='softmax')  # 26 letters + 10 digits
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


## OCR using EasyOCR

In [ ]:
import easyocr

reader = easyocr.Reader(['en'])

result = reader.readtext(plate)

for detection in result:
    print("Detected Text:", detection[1])


## Save Model

In [ ]:
model.save('anpr_model.h5')

print("ANPR model saved successfully.")


## Conclusion
This project demonstrates:
- Number plate detection using OpenCV Haar Cascade
- CNN model for character recognition
- OCR using EasyOCR